In [ ]:
# Kill all processes on the GPU
!fuser -v /dev/nvidia* -k

# Libraries

In [ ]:
%%capture
!uv pip uninstall torchao torchaudio torchvision -y
!uv pip install \
    "transformers==5.17.0" \
    "peft==0.21.0" \
    "trl" \
    "accelerate" \
    "bitsandbytes" \
    "wandb" \
    "evaluate"

In [ ]:
import os
import math
import random
import json
import logging
from datetime import datetime
from typing import Tuple, Optional
from pathlib import Path

import torch
import torch.nn as nn
from huggingface_hub import snapshot_download
import wandb
import evaluate
from transformers import (
    AutoModelForQuestionAnswering,
    AutoModelForMaskedLM,
    DataCollatorForLanguageModeling, 
    DataCollatorWithPadding,
    AutoTokenizer,
    Trainer, 
    TrainingArguments,
    EarlyStoppingCallback,
    TrainerCallback,
    TrainerState,
)
from peft import LoraConfig
from safetensors.torch import load_file, save_file
from datasets import load_dataset, Dataset

# Configuration

In [ ]:
# Run configuration
TASK = 'squad' # 'wikipedia' | 'squad'
LANG = 'en'    # wikipedia: 'en' | '<any_lang>', squad: 'en' only
SEED = 42
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# Data configuration
MLM_PROB = 0.15
# --- 3.5K (80%:20%) ---
# TRAIN_SIZE = 3_500
# VAL_SIZE = 875
# --- 15K (80%:20%) ---
TRAIN_SIZE = 15_000
VAL_SIZE = 3_750

# Model configuration
MODEL_ID = 'FacebookAI/xlm-roberta-base'
MODEL_NAME = 'XLM-R-Base'
# --- New training ---
RESUME_MODEL_ID = None
RESUME_CKPT_STEP = None
# --- Resume training ---
# RESUME_MODEL_ID = 'alxxtexxr/XLM-R-Base-squad-en-15K-s42-LegameX-LoRA-v261007201619'
# RESUME_CKPT_STEP = 4200

# LegameX configuration
# --- Reference LoRA (trained on T4) ---
REF_LORA_ID = 'alxxtexxr/XLM-R-Base-wikipedia-en-2K-s42-LoRA-v260929201303'
REF_LORA_CKPT_STEP = 1000
# --- Reference LoRA (trained on RTX 3090) ---
# REF_LORA_ID = 'alxxtexxr/XLM-R-Base-wikipedia-en-2K-s42-LoRA-v260922211643'
# REF_LORA_CKPT_STEP = 600
# --- Gate ---
GATE_RANK = 1
GATE_ALPHA = 1
GATE_WARMUP_STEPS = 0
# --- Gate offset ---
# gate_offset_total(t) = gate_offset_scheduled(t) + gate_offset_learned
# - scheduled: anneal GATE_OFFSET_INIT -> 0 over GATE_OFFSET_STEPS steps
#              (uses the same curve as gate_scheduler_type)
# - learned:   gate.offset is an nn.Parameter trained by gradient descent, init 0.0
GATE_OFFSET_INIT = -4.0
GATE_OFFSET_STEPS = 0 # anneal length; 0 keeps the scheduled term constant at GATE_OFFSET_INIT (no annealing)

# Training configuration
MINI_BATCH_SIZE = 4
GRAD_ACCUM_STEPS = 4
LR = 1e-4
# --- wikipedia, 2~4K ---
# WARMUP_STEPS = 50
# EVAL_STEPS = 100
# SAVE_STEPS = 100
# LOGGING_STEPS = 50
# ---
# NUM_EPOCHS = 3
# EARLY_STOPPING_PATIENCE = 3
# ---
# NUM_EPOCHS = 10
# EARLY_STOPPING_PATIENCE = 5
# ---
# NUM_EPOCHS = 5
# EARLY_STOPPING_PATIENCE = 5
# --- squad, 15K ---
WARMUP_STEPS = 100
EVAL_STEPS = 300
SAVE_STEPS = 300
LOGGING_STEPS = 100
# ---
# NUM_EPOCHS = 5
# EARLY_STOPPING_PATIENCE = 3
# ---
# NUM_EPOCHS = 5
# EARLY_STOPPING_PATIENCE = 5
# ---
NUM_EPOCHS = 10
EARLY_STOPPING_PATIENCE = 5
# ---
# NUM_EPOCHS = 15
# EARLY_STOPPING_PATIENCE = 10

In [ ]:
# Resume training configuration
resume_from_checkpoint = None
if RESUME_MODEL_ID:
    model_name = RESUME_MODEL_ID
    run_name = model_name.split('/')[-1]
    hub_model_id = RESUME_MODEL_ID
    
    from huggingface_hub import snapshot_download
    snapshot_download(repo_id=hub_model_id, local_dir=model_name)
    
    if RESUME_CKPT_STEP:
        resume_from_checkpoint = f"{hub_model_id}/checkpoint-{RESUME_CKPT_STEP}"
        # Ensure the checkpoint exists
        assert os.path.exists(resume_from_checkpoint), f"Checkpoint {resume_from_checkpoint} does not exist."
else:
    run_name = f'{MODEL_NAME}-{TASK}-{LANG}-{TRAIN_SIZE/1000:g}K-s{SEED}-LegameX-LoRA-v{datetime.now().strftime("%y%m%d%H%M%S")}'
    
    # Automatically retrieve the username of the currently logged-in user
    from huggingface_hub import HfApi
    user_info = HfApi().whoami()
    username = user_info["name"]

    hub_model_id = f"{username}/{run_name}"
base_hub_model_id, version = hub_model_id.rsplit('-v', 1)
hub_merged_model_id = f'{base_hub_model_id}-mrg-v{version}'

print("Resume from checkpoint:", resume_from_checkpoint)
print("Model name:", MODEL_NAME)
print("Run name:", run_name)
print("Hub model ID:", hub_model_id)
print("Hub merged model ID:", hub_merged_model_id)

# Utilities

In [ ]:
# LoRA utilities
def download_hf_model(
        repo_id: str, 
        ckpt_step: int, 
        max_checkpoints: int = 10_000,
        ckpt_interval: int = 25,
    ) -> Tuple[Path, Optional[Path]]:
    assert len(repo_id.split("/")) == 2, f"Invalid repo_id format: {repo_id}. Expected format: 'username/repo_name'."
    local_dir = Path(repo_id.split('/')[-1])
    ignore_checkpoints = None
    
    if ckpt_step is not None:
        ignore_checkpoints = [f'checkpoint-{i}/*' for i in range(0, max_checkpoints, ckpt_interval) if i != ckpt_step]
        ignore_checkpoints += ['runs/*', '**/wandb/*']

    snapshot_download(
        repo_id=repo_id,
        local_dir=local_dir,
        ignore_patterns=ignore_checkpoints,
    )

    ckpt_dir = None
    if ckpt_step is not None:
        ckpt_dir = local_dir / f'checkpoint-{ckpt_step}'
    return local_dir, ckpt_dir

def check_parameter(n, p):
    print(f"{'name':<8}:", n)
    print(f"{'device':<8}:", p.device)
    print(f"{'dtype':<8}:", p.dtype)
    print(f"{'norm':<8}:", p.norm().item())

def check_weights(model, infix, limit=1):
    for i, (n, p) in enumerate(model.named_parameters()):
        if infix in n:
            check_parameter(n, p)
            print()
            if i >= limit - 1:
                break

# Data utilities
def load_train_val_datasets(
    task, # 'wikipedia' | 'squad'
    lang, # e.g., 'en' | 'ja' | 'id'
    train_size, 
    val_size,
    token=None,
):
    # Validate that if the task is 'squad', the language must be 'en'
    if task == 'squad':
        assert lang == 'en', "SQuAD is English-only."
    
    # Define dataset configurations for each task
    data_configs = {
        'wikipedia': {
            'data_id': 'wikimedia/wikipedia',
            'data_dir': f'20231101.{lang}',
            'train_split': 'train',
            'val_split': 'train',
        },
        'squad': {
            'data_id': 'rajpurkar/squad',
            'data_dir': None,
            'train_split': 'train',
            'val_split': 'validation',
        },
    }
    
    # Validate that the specified task is supported
    assert task in data_configs, (
        f"Unsupported task: {task}. "
        f"Supported tasks: {list(data_configs.keys())}"
    )

    # Set up Hugging Face dataset configuration
    data_id = data_configs[task]['data_id']
    data_dir = data_configs[task]['data_dir']
    train_split = data_configs[task]['train_split']
    val_split = data_configs[task]['val_split']

    if train_split == val_split:
        # If the train and validation splits are the same, we need to sample from the same dataset stream
        dataset_stream = load_dataset(
            data_id,
            data_dir=data_dir,
            split=train_split,
            streaming=True,
            token=token,
        )

        train_data = []
        val_data = []

        for i, example in enumerate(dataset_stream):
            if i < train_size:
                train_data.append(example)
            elif i < train_size + val_size:
                val_data.append(example)
            else:
                break

    else:
        # If the train and validation splits are different, we can sample from each split separately
        def sample_split(split, size):
            dataset_stream = load_dataset(
                data_id,
                data_dir=data_dir,
                split=split,
                streaming=True,
            )

            data = []
            for i, example in enumerate(dataset_stream):
                if i >= size:
                    break
                data.append(example)
            return data

        train_data = sample_split(train_split, train_size)
        val_data = sample_split(val_split, val_size)

    return (
        Dataset.from_list(train_data),
        Dataset.from_list(val_data),
    )

# Model

In [ ]:
# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# Determine the appropriate model classes based on the task
if TASK == 'squad':
    model_cls = AutoModelForQuestionAnswering
    task_type = 'QUESTION_ANS'
    data_collator = DataCollatorWithPadding(tokenizer)
    label_names = ['start_positions', 'end_positions']
else:
    model_cls = AutoModelForMaskedLM
    task_type = 'FEATURE_EXTRACTION'
    data_collator = DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=True,
        mlm_probability=MLM_PROB,
    )
    label_names = ['labels']

# Load the base model
base_model = model_cls.from_pretrained(MODEL_ID)

In [ ]:
import torch.nn.functional as F

# GATE_INIT_OFFSET = -6.0
# GATE_INIT_OFFSET=-6 initializes the gate output close to 0 while keeping it in (0, 1).
# sigmoid(-6) ≈ 0.0025 and sigmoid'(-6) ≈ 0.0025, so the gradient is small but nonzero, 
# allowing the gate to grow if the loss favors it.
# GATE_INIT_OFFSET = -4.0


class LegamexLora(nn.Module):
    def __init__(
            self, base_module, 
            
            # Reference (ref) and transfer (tfr) LoRA parameters
            ref_lora_config, tfr_lora_config,
                    
            # Gate parameters
            gate_warmup_steps, gate_scheduler_type='cosine', 
            gate_rank=1, gate_alpha=1, gate_dropout=0.0, 
            gate_bias=False, gate_use_rslora=False,
            
            # Gate offset parameters
            # gate_offset_total(t) = gate_offset_scheduled(t) + gate_offset_learned
            gate_offset_init=-4.0,   # scheduled term: anneals to 0.0 over gate_offset_steps
            gate_offset_steps=0,     # 0 keeps the scheduled term constant at gate_offset_init
        ):
        super().__init__()
        
        assert ref_lora_config.lora_dropout == 0.0, "Weight-blend gate doesn't support dropout"
        assert tfr_lora_config.lora_dropout == 0.0, "Weight-blend gate doesn't support dropout"
        assert not gate_bias, "Weight-blend gate doesn't use bias"
        
        assert gate_scheduler_type in ['linear', 'cosine', 'quadratic', 'cubic', 'sqrt'], (
            f"Unknown gate scheduler type: {gate_scheduler_type}. "
            "Must be one of ['linear', 'cosine', 'quadratic', 'cubic', 'sqrt']."
        )
        
        self.base_module = base_module
        self.gate_warmup_steps = gate_warmup_steps
        self.gate_scheduler_type = gate_scheduler_type
        self.training_step = 0
        self.gate_offset_init = gate_offset_init
        self.gate_offset_steps = gate_offset_steps
        
        # Determine in_features and out_features from the base module
        in_features = getattr(base_module, 'in_features', None)
        out_features = getattr(base_module, 'out_features', None)
        if in_features is None or out_features is None:
            raise ValueError(f"Cannot determine in_features or out_features from {base_module}.")
        
        # Initialize the reference (ref) and transfer (tfr) LoRA modules
        self.lora = nn.ModuleDict({
            'ref': self._init_lora_module(in_features, out_features, 
                                          rank=ref_lora_config.r, 
                                          alpha=ref_lora_config.lora_alpha, 
                                          dropout=ref_lora_config.lora_dropout, 
                                          bias=False if ref_lora_config.bias == 'none' else True, 
                                          use_rslora=ref_lora_config.use_rslora),
            'tfr': self._init_lora_module(in_features, out_features, 
                                          rank=tfr_lora_config.r, 
                                          alpha=tfr_lora_config.lora_alpha, 
                                          dropout=tfr_lora_config.lora_dropout, 
                                          bias=False if tfr_lora_config.bias == 'none' else True, 
                                          use_rslora=tfr_lora_config.use_rslora),
        })
        
        # Initialize the gate modules
        self.gate = self._init_lora_module(in_features, out_features, 
                                           rank=gate_rank, alpha=gate_alpha, 
                                           dropout=gate_dropout, bias=gate_bias, 
                                           use_rslora=gate_use_rslora)
        # gate_offset_learned: initialized to 0.0; the anneal start (e.g. -4.0)
        # lives in the scheduled term, gate_offset_scheduler()
        self.gate.offset = nn.Parameter(
            torch.tensor(
                0.0,
                dtype=self.gate.B.weight.dtype,   # type: ignore
                device=self.gate.B.weight.device, # type: ignore
            )
        )


    def _init_lora_module(self, in_features, out_features, rank, alpha, dropout, bias, use_rslora):
        device = self.base_module.weight.device
        dropout_module = nn.Dropout(dropout) if dropout > 0.0 else nn.Identity()
        scaling = alpha / math.sqrt(rank) if use_rslora else alpha / rank
        
        # Initialize the LoRA modules with the extracted parameters
        lora_A = nn.Linear(in_features, rank, bias=bias, device=device)
        lora_B = nn.Linear(rank, out_features, bias=bias, device=device)
        
        # Initialize the weights for the LoRA modules
        # nn.init.normal_(lora_A.weight, mean=0.0, std=1 / math.sqrt(rank))
        nn.init.kaiming_uniform_(lora_A.weight, a=math.sqrt(5))
        nn.init.zeros_(lora_B.weight)
        
        lora_module = nn.ModuleDict({
            'A': lora_A,
            'B': lora_B,
            'dropout': dropout_module,
        })
        lora_module.scaling = scaling
        lora_module.use_bias = bias
        return lora_module


    def load_lora_weights(self, state_dict, prefix, lora_type):
        assert lora_type in ['ref', 'tfr'], "Invalid LoRA type. Must be 'ref' or 'tfr'."
        
        device_A = self.lora[lora_type].A.weight.device # type: ignore
        device_B = self.lora[lora_type].B.weight.device # type: ignore

        key_A = f'{prefix}.lora_A.weight'
        key_B = f'{prefix}.lora_B.weight'

        with torch.no_grad():
            self.lora[lora_type].A.weight.copy_(state_dict[key_A].to(device_A)) # type: ignore
            self.lora[lora_type].B.weight.copy_(state_dict[key_B].to(device_B)) # type: ignore

            if self.lora[lora_type].use_bias:
                if f'{prefix}.lora_A.bias' in state_dict:
                    self.lora[lora_type].A.bias.copy_(state_dict[f'{prefix}.lora_A.bias'].to(device_A)) # type: ignore
                if f'{prefix}.lora_B.bias' in state_dict:
                    self.lora[lora_type].B.bias.copy_(state_dict[f'{prefix}.lora_B.bias'].to(device_B)) # type: ignore    


    @staticmethod
    def _scheduler_progress(step, num_steps, scheduler_type):
        """Interpolation factor in [0, 1] following scheduler_type over num_steps."""
        if num_steps <= 0:
            return 1.0
        
        t = min(max(step / num_steps, 0.0), 1.0)
        
        if scheduler_type == 'linear':
            p = t
        elif scheduler_type == 'cosine':
            # cosine warmup
            p = 0.5 * (1.0 - math.cos(math.pi * t))
        elif scheduler_type == 'quadratic':
            p = t**2
        elif scheduler_type == 'cubic':
            p = t**3
        elif scheduler_type == 'sqrt':
            p = math.sqrt(t)
        else:
            raise ValueError(
                f"Unknown gate scheduler type: {scheduler_type}. "
                "Must be one of ['linear', 'cosine', 'quadratic', 'cubic', 'sqrt']."
            )
        
        return p


    def gate_scheduler(self):
        # Warm the gate output from 0.0 to 1.0 over gate_warmup_steps
        p = self._scheduler_progress(self.training_step, self.gate_warmup_steps, self.gate_scheduler_type)
        return 0.0 + (1.0 - 0.0) * p


    def gate_offset_scheduler(self):
        # Anneal the scheduled gate offset from gate_offset_init to 0.0 over
        # gate_offset_steps, using the same curve as gate_scheduler_type.
        # E.g. gate_offset_init=-4.0 -> 0.0: the gate sigmoid bias starts at
        # sigmoid(-4) ≈ 0.018 (gate ≈ 0 at init) and relaxes to sigmoid(0) = 0.5,
        # so the gate is free to open as training progresses.
        # gate_offset_steps <= 0 means NO annealing: the scheduled term stays at
        # gate_offset_init for the whole run, so gate_offset_total always starts
        # at sigmoid(gate_offset_init) no matter how the learned term moves.
        if self.gate_offset_steps <= 0:
            return float(self.gate_offset_init)
        p = self._scheduler_progress(self.training_step, self.gate_offset_steps, self.gate_scheduler_type)
        return float(self.gate_offset_init) + (0.0 - float(self.gate_offset_init)) * p


    def current_gate_offset(self):
        """Return the total gate offset b_g for the current training step:

        gate_offset_total(t) = gate_offset_scheduled(t) + gate_offset_learned

        - gate_offset_scheduled(t): gate_offset_scheduler() -> gate_offset_init
          annealed to 0.0 over gate_offset_steps (constant gate_offset_init when
          steps <= 0) (float)
        - gate_offset_learned:      the gate.offset nn.Parameter (tensor, init 0.0,
          receives gradients)
        """
        return self.gate_offset_scheduler() + self.gate.offset


    def forward(self, x):
        # Compute the base layer output
        base_out = self.base_module(x)
        
        # Check if the input tensor needs to be converted to the LoRA module's dtype
        requires_conversion = not torch.is_autocast_enabled()
        if requires_conversion:
            # TODO: Add assertion to check if all weights have the same dtype
            x = x.to(self.lora.tfr.A.weight.dtype) # type: ignore
        
        # --- OLD ---
        # # Compute the outputs for the reference and transfer LoRA
        # ref_lora_out = self.lora.ref.B(self.lora.ref.dropout(self.lora.ref.A(x))) * self.lora.ref.scaling # type: ignore
        # tfr_lora_out = self.lora.tfr.B(self.lora.tfr.dropout(self.lora.tfr.A(x))) * self.lora.tfr.scaling # type: ignore
        
        # # Compute the gate output and its complement
        # # gate_weight = torch.tanh(self.gate.B.weight @ self.gate.A.weight) # type: ignore
        # # gate_weight = torch.clamp(self.gate.B.weight @ self.gate.A.weight, min=0.0, max=1.0) # type: ignore
        # gate_weight = torch.sigmoid((self.gate.B.weight @ self.gate.A.weight) - 6.0) # type: ignore
        # # -6 initializes the gate output close to 0 while keeping it in (0, 1).
        # # sigmoid(-6) ≈ 0.0025 and sigmoid'(-6) ≈ 0.0025, so the gradient is small but nonzero, 
        # # allowing the gate to grow if the loss favors it.
        
        # if self.gate.use_bias:
        #     gate_bias = self.gate.B.bias + self.gate.B.weight @ self.gate.A.bias # type: ignore
        # else:
        #     gate_bias = 0
        
        # # gate_out = x @ gate_weight.T + gate_bias
        # gate_out = gate_weight.T + gate_bias
        # gate_out_scheduled = gate_out * self.gate_scheduler() # Schedule the gate output based on the training step and warmup steps
        # gate_comp_out = 1 - gate_out_scheduled
        
        # # Compute the LoRA output
        # lora_out = ref_lora_out * gate_out_scheduled + tfr_lora_out * gate_comp_out
        # --- NEW ---
        # Effective [out, in] matrices
        W_ref = self.lora.ref.B.weight @ self.lora.ref.A.weight # type: ignore # [out, in]
        W_tfr = self.lora.tfr.B.weight @ self.lora.tfr.A.weight # type: ignore # [out, in]
        W_g = self.gate.B.weight @ self.gate.A.weight           # type: ignore # [out, in]
        
        # Gate offset: scheduled anneal (gate_offset_init -> 0 over gate_offset_steps)
        # plus the learned gate.offset parameter. See current_gate_offset() for details.
        b_g = self.current_gate_offset()

        # Per-weight gate, ~0 at init
        g = torch.sigmoid(W_g + b_g)  # type: ignore
        g = g * self.gate_scheduler() # OPTIONAL: Schedule the gate output based on the training step and warmup steps

        # Blend the deltas (still [out, in])
        # W_blend = g * (W_ref * self.lora.ref.scaling) + (1.0 - g) * (W_tfr * self.lora.tfr.scaling) # type: ignore
        W_blend = (1.0 - g) * (W_ref * self.lora.ref.scaling) + g * (W_tfr * self.lora.tfr.scaling) # type: ignore

        # Apply as a single linear
        lora_out = F.linear(x, W_blend) # [..., out]
        # ---
        
        # Check if the LoRA output needs to be converted back to the base layer's dtype
        if requires_conversion:
            lora_out = lora_out.to(base_out.dtype)
            
        return base_out + lora_out


    def set_training_step(self, training_step):
        self.training_step = training_step

In [ ]:
class LegamexLoraModel(nn.Module):
    def __init__(
            self, base_model, task_type, 
            
            # Reference (ref) and transfer (tfr) LoRA parameters
            ref_lora_config, tfr_lora_config, 
            
            # Gate parameters
            gate_warmup_steps, gate_scheduler_type="cosine", 
            gate_rank=1, gate_alpha=1, gate_dropout=0.0, 
            gate_bias=False, gate_use_rslora=False,
            
            # Gate offset parameters
            # gate_offset_total(t) = gate_offset_scheduled(t) + gate_offset_learned
            gate_offset_init=-4.0,   # scheduled term: anneals to 0.0 over gate_offset_steps
            gate_offset_steps=0,     # 0 keeps the scheduled term constant at gate_offset_init
        ):
        super().__init__()
        
        assert ref_lora_config.bias == 'none', "Weight-blend gate requires bias='none'"
        assert tfr_lora_config.bias == 'none', "Weight-blend gate requires bias='none'"
        
        # Determine the base model name or path
        self.base_model_name_or_path = (
            ref_lora_config.base_model_name_or_path
            if hasattr(ref_lora_config, 'base_model_name_or_path')
            else base_model.config.name_or_path
        )
        if base_model.config.name_or_path != self.base_model_name_or_path:
            print("Using the base model name from the reference LoRA:", self.base_model_name_or_path)
        
        # Store the parameters
        self.base_model = base_model
        self.task_type = task_type
        self.ref_lora_config = ref_lora_config
        self.tfr_lora_config = tfr_lora_config
        self.gate_warmup_steps = gate_warmup_steps
        self.gate_scheduler_type = gate_scheduler_type
        self.gate_rank = gate_rank
        self.gate_alpha = gate_alpha
        self.gate_dropout = gate_dropout
        self.gate_bias = gate_bias
        self.gate_use_rslora = gate_use_rslora
        self.gate_offset_init = gate_offset_init
        self.gate_offset_steps = gate_offset_steps

        # Wrap the target modules with LegameX
        self.legamex_modules = {}
        self._wrap_target_modules_with_legamex()


    def _get_parent_module(self, module_name):
            parts = module_name.split('.')
            parent_module = self.base_model
            for part in parts[:-1]:
                parent_module = getattr(parent_module, part)
            return parent_module, parts[-1]


    def _wrap_target_modules_with_legamex(self):
        assert self.ref_lora_config.target_modules == self.tfr_lora_config.target_modules, \
            "Target modules for reference and transfer LoRA must be the same."
        target_modules = self.tfr_lora_config.target_modules
        
        for module_name, module in self.base_model.named_modules():
            # Skip the modules that are already wrapped with LegamexLora
            if isinstance(module, LegamexLora):
                module_name = module_name.replace('.', '__DOT__')
                self.legamex_modules[module_name] = module
                continue
            
            if any(target_module in module_name for target_module in target_modules):
                # Wrap the module with LegamexLora
                parent_module, child_name = self._get_parent_module(module_name)
                legamex_module = LegamexLora(
                    base_module=module,
                    ref_lora_config=self.ref_lora_config,
                    tfr_lora_config=self.tfr_lora_config,
                    gate_warmup_steps=self.gate_warmup_steps,
                    gate_scheduler_type=self.gate_scheduler_type,
                    gate_rank=self.gate_rank,
                    gate_alpha=self.gate_alpha,
                    gate_dropout=self.gate_dropout,
                    gate_bias=self.gate_bias,
                    gate_use_rslora=self.gate_use_rslora,
                    gate_offset_init=self.gate_offset_init,
                    gate_offset_steps=self.gate_offset_steps,
                )
                setattr(parent_module, child_name, legamex_module)
                
                # Store the wrapped module
                module_name = module_name.replace('.', '__DOT__')
                self.legamex_modules[module_name] = legamex_module
                
            # Store the qa_outputs modules if task_type is 'QUESTION_ANS'
            if isinstance(module, nn.Linear) and 'qa_outputs' in module_name and self.task_type == 'QUESTION_ANS':
                module_name = module_name.replace('.', '__DOT__')
                self.legamex_modules[module_name] = module
                
        # Freeze all parameters in the base model and unfreeze only the LegameX modules except for the reference LoRA
        self.freeze_all(verbose=False)
        self.unfreeze_legamex_except_ref_lora(verbose=True)


    def _json_safe(self, obj):
        if isinstance(obj, set):
            return list(obj)
        if isinstance(obj, dict):
            return {k: self._json_safe(v) for k, v in obj.items()}
        if isinstance(obj, list):
            return [self._json_safe(v) for v in obj]
        return obj


    def freeze_all(self, verbose=False):
        for p in self.base_model.parameters():
            p.requires_grad = False
        if verbose:
            print("All modules are frozen.")


    def unfreeze_legamex_except_ref_lora(self, verbose=False):
        for module_name, module in self.legamex_modules.items():
            if isinstance(module, LegamexLora):
                # Unfreeze the transfer LoRA weights and biases
                module.lora.tfr.A.weight.requires_grad = True # type: ignore
                module.lora.tfr.B.weight.requires_grad = True # type: ignore
                if module.lora.tfr.use_bias: # type: ignore
                    if hasattr(module.lora.tfr.A, 'bias') and module.lora.tfr.A.bias is not None: # type: ignore
                        module.lora.tfr.A.bias.requires_grad = True # type: ignore
                    if hasattr(module.lora.tfr.B, 'bias') and module.lora.tfr.B.bias is not None: # type: ignore
                        module.lora.tfr.B.bias.requires_grad = True # type: ignore
                
                # Unfreeze the gate weights and biases
                module.gate.A.weight.requires_grad = True # type: ignore
                module.gate.B.weight.requires_grad = True # type: ignore
                module.gate.offset.requires_grad = True # type: ignore
                if module.gate.use_bias:
                    if hasattr(module.gate.A, 'bias') and module.gate.A.bias is not None: # type: ignore
                        module.gate.A.bias.requires_grad = True # type: ignore
                    if hasattr(module.gate.B, 'bias') and module.gate.B.bias is not None: # type: ignore
                        module.gate.B.bias.requires_grad = True # type: ignore

            # Unfreeze the qa_outputs weight and bias if task_type is 'QUESTION_ANS'
            elif isinstance(module, nn.Linear) and 'qa_outputs' in module_name and self.task_type == 'QUESTION_ANS':    
                module.weight.requires_grad = True
                if hasattr(module, 'bias') and module.bias is not None:
                        module.bias.requires_grad = True

        if verbose:
            trainable = sum(p.numel() for p in self.base_model.parameters() if p.requires_grad)
            total = sum(p.numel() for p in self.base_model.parameters())
            print(f"Trainable parameters: {trainable:,} / {total:,}")


    def load_lora_weights(self, lora_path: str, lora_type: str):
        assert isinstance(lora_path, str), "LoRA weights file path must be a string."
        assert lora_type in ['ref', 'tfr'], "Invalid LoRA type. Must be 'ref' or 'tfr'."
        
        if not os.path.exists(lora_path):
            raise FileNotFoundError(f"LoRA weights file not found: {lora_path}")
        
        if lora_path.endswith('.safetensors'):
            state_dict = load_file(lora_path)
        else:
            state_dict = torch.load(lora_path, map_location='cpu')
            
        processed_module_names = set()
        for key in state_dict.keys():
            key_prefix, key_rest = key.rsplit('model.', 1)
            module_name = key_rest.split('.lora', 1)[0]
            module_name_sanitized = module_name.replace('.', '__DOT__')
            
            if module_name_sanitized in processed_module_names:
                continue
            
            if module_name_sanitized in self.legamex_modules:
                prefix = key_prefix + 'model.' + module_name
                self.legamex_modules[module_name_sanitized].load_lora_weights(state_dict, prefix, lora_type)
            else:
                print("LoRA weights not found for module:", module_name)
                
            processed_module_names.add(module_name_sanitized)

        print(f"LoRA weights loaded from {lora_path} into model.")


    def load_legamex_state_dict(self, save_dir: str):
        """Copy legamex.pt weights from save_dir INTO this (already built) instance.

        The resume path must load weights in-place: the optimizer passed to the
        Trainer holds references to THIS instance's parameter tensors, so the
        instance must never be swapped for a reconstructed one (see
        LegamexTrainer._load_from_checkpoint for why). from_pretrained() also
        uses this method after rebuilding the architecture, so the loading
        logic exists exactly once.
        """
        state_dict_path = os.path.join(save_dir, 'legamex.pt')
        legamex_state_dict = torch.load(state_dict_path, map_location='cpu')

        n_loaded = 0
        unmatched = []
        for k, v in legamex_state_dict.items():
            parts = k.split('.', 1) # ['module_name', '<rest_of_path>']
            if len(parts) != 2:
                unmatched.append(k)
                continue
            module_name, param_name = parts
            module = self.legamex_modules.get(module_name)
            param = None if module is None else dict(module.named_parameters()).get(param_name)
            if param is None:
                unmatched.append(k)
                continue
            param.data.copy_(v.to(param.device))
            n_loaded += 1

        if n_loaded == 0:
            raise RuntimeError(
                f"No LegameX parameters from {state_dict_path} matched this model. "
                "Check that the checkpoint belongs to the same architecture "
                "(task type, target modules, gate rank)."
            )
        if unmatched:
            print(f"Warning: {len(unmatched)} checkpoint key(s) did not match any parameter, e.g.: {unmatched[:3]}")
        print(f"Loaded {n_loaded} LegameX parameter tensors from {state_dict_path} into the existing model.")


    @classmethod
    def from_pretrained(cls, save_dir: str, **kwargs):
        # Load the LegameX configuration
        with open(os.path.join(save_dir, 'legamex_config.json'), 'r') as f:
            config = json.load(f)
            
        # Re-create the base model
        base_model_id = config['base_model_name_or_path']
        task_type = config['task_type']
        if task_type == 'QUESTION_ANS':
            base_model = AutoModelForQuestionAnswering.from_pretrained(base_model_id)
        elif task_type == 'FEATURE_EXTRACTION':
            base_model = AutoModelForMaskedLM.from_pretrained(base_model_id)
        else:
            raise ValueError(f"Unknown task type: {task_type}")
        
        # Load the reference and transfer LoRA configurations
        ref_lora_config = LoraConfig(**config['ref_lora_config'])
        tfr_lora_config = LoraConfig(**config['tfr_lora_config'])
        
        # Re-create the LegameX model
        model = cls(
            base_model=base_model,
            task_type=task_type,
            ref_lora_config=ref_lora_config,
            tfr_lora_config=tfr_lora_config,
            gate_warmup_steps=config['gate_warmup_steps'],
            gate_scheduler_type=config['gate_scheduler_type'],
            gate_rank=config['gate_rank'],
            gate_alpha=config['gate_alpha'],
            gate_dropout=config['gate_dropout'],
            gate_bias=config['gate_bias'],
            gate_use_rslora=config['gate_use_rslora'],
            # Gate offset (config.get keeps old checkpoints loadable; the removed
            # gate_offset_type key found in old configs is simply ignored)
            gate_offset_init=config.get('gate_offset_init', -4.0),
            gate_offset_steps=config.get('gate_offset_steps', 0),
        )
        
        # Load the LegameX state dict into the freshly rebuilt instance
        model.load_legamex_state_dict(save_dir)

        return model


    def save_pretrained(self, save_dir):
        # Create save directories
        os.makedirs(save_dir, exist_ok=True)
        # tfr_dir = os.path.join(save_dir, 'tfr')
        # os.makedirs(tfr_dir, exist_ok=True)
        
        # Store the LegameX and transfer component state dicts
        state_dict = {}
        # tfr_state_dict = {}
        
        for module_name, module in self.legamex_modules.items():
            for param_name, param in module.named_parameters():
                # Skip the base_module parameters
                if 'base_module' in param_name:
                    continue
                
                # Store the rest of parameters in the LegameX state dict
                param_name_full = f'{module_name}.{param_name}'
                param_cpu = param.detach().cpu()
                state_dict[param_name_full] = param_cpu
                
                # # Store the transfer component and qa_outputs parameters separately
                # if 'lora.tfr' in param_name or 'qa_outputs' in module_name:
                #     param_name_full_sanitized = param_name_full.replace('__DOT__', '.').replace('lora.tfr.', 'lora_')
                #     tfr_param_cpu = param_cpu

                #     # If the parameter is a transfer component weight, apply the gate complement weight to it
                #     if 'lora.tfr' in param_name and 'weight' in param_name:
                #         gate_weight_name = param_name.replace('lora.tfr', 'gate')
                        
                #         try:
                #             gate_param = module.get_parameter(gate_weight_name)
                #             gate_weight = gate_param.detach().cpu()
                #             gate_comp_weight = 1.0 - gate_weight
                #             tfr_param_cpu = tfr_param_cpu * gate_comp_weight
                #         except AttributeError:
                #             raise AttributeError(
                #                 f"Could not find matching gate weight component '{gate_weight_name}' "
                #                 f"in module '{module_name}' to pair with transfer component weights."
                #             )
                        
                #     tfr_state_dict[param_name_full_sanitized] = tfr_param_cpu
                    
        torch.save(state_dict, os.path.join(save_dir, 'legamex.pt'))
        
        # Save the transfer component state dict and its configuration
        # save_file(tfr_state_dict, os.path.join(tfr_dir, 'adapter_model.safetensors'))
        # with open(os.path.join(tfr_dir, 'adapter_config.json'), 'w') as f:
        #     json.dump(self._json_safe(self.tfr_lora_config.to_dict()), f, indent=4)
        
        # Save the LegameX configuration for rebuilding wrapper
        config = {
            'base_model_name_or_path': self.base_model_name_or_path,
            'task_type': self.task_type,
            'ref_lora_config': self._json_safe(self.ref_lora_config.to_dict()),
            'tfr_lora_config': self._json_safe(self.tfr_lora_config.to_dict()),
            'gate_warmup_steps': self.gate_warmup_steps,
            'gate_scheduler_type': self.gate_scheduler_type,
            'gate_rank': self.gate_rank,
            'gate_alpha': self.gate_alpha,
            'gate_dropout': self.gate_dropout,
            'gate_bias': self.gate_bias,
            'gate_use_rslora': self.gate_use_rslora,
            'gate_offset_init': self.gate_offset_init,
            'gate_offset_steps': self.gate_offset_steps,
        }
        with open(os.path.join(save_dir, 'legamex_config.json'), 'w') as f:
            json.dump(config, f, indent=4)


    def merge_and_unload(self):
        self.eval() # Ensure the dropout in the LoRA branches is disabled

        # Warn if any gate is not fully warmed up yet (gate_scheduler() < 1.0),
        # or if the scheduled gate offset has not fully annealed to 0.0 yet
        # (only meaningful when annealing is configured, i.e. gate_offset_steps > 0;
        # with steps <= 0 the scheduled term intentionally stays at gate_offset_init)
        cold_gates = [
            name for name, module in self.legamex_modules.items()
            if isinstance(module, LegamexLora) and (
                module.gate_scheduler() < 1.0
                or (module.gate_offset_steps > 0 and module.gate_offset_scheduler() != 0.0)
            )
        ]
        if cold_gates:
            print(
                f"Warning: {len(cold_gates)} gate(s) are not fully warmed up or their "
                f"scheduled offset has not annealed to 0 (e.g. {cold_gates[0]}); "
                "the merge assumes gate_scheduler() == 1.0 and gate_offset_scheduled == 0."
            )

        with torch.no_grad():
            for module_name, module in self.legamex_modules.items():
                # The qa_outputs head is already trained in the base model; nothing to merge
                if not isinstance(module, LegamexLora):
                    continue

                # Restore the real module name and locate its parent in the base model
                real_module_name = module_name.replace('__DOT__', '.')
                parent_module, child_name = self._get_parent_module(real_module_name)

                # Get the base module, reference LoRA, transfer LoRA, and gate components
                base_module: nn.Module = module.base_module
                ref = module.lora.ref
                tfr = module.lora.tfr
                gate = module.gate
                
                # # Merge the reference and transfer LoRA biases if enabled
                # if ref.use_bias: # type: ignore
                #     assert base_module.bias is not None
                #     raise NotImplementedError("Merging the reference LoRA bias is not implemented yet.")
                # if tfr.use_bias: # type: ignore
                #     assert base_module.bias is not None
                #     raise NotImplementedError("Merging the transfer LoRA bias is not implemented yet.")
                
                # Compute the gate weights and their complement
                # gate_W = torch.tanh(gate.B.weight @ gate.A.weight) # type: ignore
                # gate_W = torch.clamp(gate.B.weight @ gate.A.weight, min=0.0, max=1.0) # type: ignore
                # gate_W = torch.sigmoid((gate.B.weight @ gate.A.weight) + GATE_INIT_OFFSET) # type: ignore
                gate_offset = module.current_gate_offset() # scheduled + learned
                gate_W = torch.sigmoid((gate.B.weight @ gate.A.weight) + gate_offset) # type: ignore
                gate_comp_W = 1.0 - gate_W

                # Compute the reference and transfer LoRA weights
                ref_W = (ref.B.weight @ ref.A.weight) * ref.scaling # type: ignore
                tfr_W = (tfr.B.weight @ tfr.A.weight) * tfr.scaling # type: ignore

                # Merge the reference and transfer LoRA weights into the base module
                # delta_W = (ref_W * gate_W + tfr_W * gate_comp_W).to(base_module.weight.dtype) # type: ignore
                delta_W = (ref_W * gate_comp_W + tfr_W * gate_W).to(base_module.weight.dtype) # type: ignore
                base_module.weight.data += delta_W

                # Unload: replace the wrapper with the merged base module
                setattr(parent_module, child_name, base_module)

        # The model is now fully merged; drop the wrapper registry
        self.legamex_modules = {}

        return self.base_model


    def forward(self, *args, **kwargs):
            kwargs.pop('num_items_in_batch', None)
            return self.base_model.forward(*args, **kwargs)


    def set_training_step(self, training_step):
        for module in self.legamex_modules.values():
            if hasattr(module, 'set_training_step'):
                module.set_training_step(training_step)


    def __getattr__(self, name):
        try:
            return super().__getattr__(name)
        except AttributeError:
            return getattr(self.base_model, name)

In [ ]:
# Load the reference LoRA configuration
_, ref_lora_dir = download_hf_model(REF_LORA_ID, REF_LORA_CKPT_STEP)
assert ref_lora_dir is not None, "No reference LoRA directory was returned."
ref_lora_file = ref_lora_dir / 'adapter_model.safetensors'
ref_lora_config = LoraConfig.from_pretrained(str(ref_lora_dir))

# Set the reference LoRA dropout to 0.0 because it is not used
ref_lora_config.lora_dropout = 0.0 # type: ignore
print(ref_lora_config.__dict__)

In [ ]:
# Initialize the LegameX model
legamex_model = LegamexLoraModel(
    base_model,
    task_type=task_type,
    ref_lora_config=ref_lora_config,
    tfr_lora_config=ref_lora_config, # Set this to be the same as the reference LoRA configuration
    gate_warmup_steps=GATE_WARMUP_STEPS,
    gate_offset_init=GATE_OFFSET_INIT,
    gate_offset_steps=GATE_OFFSET_STEPS,
)
legamex_model.to(DEVICE);

In [ ]:
print("Check unloaded LegameX-specific weights:")
check_weights(legamex_model, infix='lora.ref')
check_weights(legamex_model, infix='lora.tfr')
check_weights(legamex_model, infix='gate')

legamex_model.load_lora_weights(str(ref_lora_file), lora_type='ref')
print()

print("Check loaded LegameX-specific weights:")
check_weights(legamex_model, infix='lora.ref')
check_weights(legamex_model, infix='lora.tfr')
check_weights(legamex_model, infix='gate')

In [ ]:
# Sanity check: ensure that the only trainable parameters are the transfer LoRA, gate, and qa_outputs weights and biases, 
# not the reference LoRA and base model weights
# for n, p in legamex_model.named_parameters():
#     print(f"{n}: {p.requires_grad}")

# Data

In [ ]:
# Load the dataset
train_dataset, val_dataset = load_train_val_datasets(
    task=TASK,
    lang=LANG, 
    train_size=TRAIN_SIZE, 
    val_size=VAL_SIZE
)

print("Training dataset:")
print(train_dataset)
print()
print("Validation dataset:")
print(val_dataset)

In [ ]:
# Preprocess the dataset
WIKI_MAX_LENGTH = 510  # 510 + 2 (BOS + EOS) = 512, matching XLM-R's max_position_embeddings
NUM_CHUNKS_PER_WIKI_ARTICLE = 3

# Bookkeeping data for string-level SQuAD evaluation
#
# IMPORTANT: datasets.map(batched=True) invokes preprocess_squad once per
# BATCH (default batch_size=1000), so the bookkeeping must be ACCUMULATED
# across batches. Overwriting it here would leave only the LAST batch's
# features, which is shorter than the full eval predictions and crashes
# compute_metrics_squad with IndexError: list index out of range.
_squad_eval_bookkeeping = {
    'feature_example_ids': [],
    'feature_offset_mappings': [],
    'feature_contexts': [],
    'references': [],
}

if TASK == 'squad':

    def preprocess_squad(examples, save_bookkeeping=False):
        tokenized = tokenizer(
            examples['question'],
            examples['context'],
            truncation='only_second', # Only truncate the context
            max_length=384,
            stride=128,
            return_overflowing_tokens=True,
            return_offsets_mapping=True,
            padding=False,
        )

        # Each generated window points back to its original SQuAD example.
        sample_mapping = tokenized.pop('overflow_to_sample_mapping') # Global index

        # Save offsets separately because the model does not need them.
        offset_mapping = tokenized.pop('offset_mapping') # Local index

        start_positions = []
        end_positions = []

        # Indices of windows that contain the complete answer.
        keep_indices = []

        for feature_idx, offsets in enumerate(offset_mapping):

            # Which original example produced this window?
            sample_idx = sample_mapping[feature_idx]

            answer = examples['answers'][sample_idx]

            answer_start_char = answer['answer_start'][0]
            answer_text = answer['text'][0]
            answer_end_char = (
                answer_start_char + len(answer_text)
            )

            # sequence_ids tells us which tokens belong to:
            # 0 = question
            # 1 = context
            # None = special tokens
            sequence_ids = tokenized.sequence_ids(feature_idx)

            # Find the first and last context tokens in this window.
            context_token_indices = [
                idx
                for idx, seq_id in enumerate(sequence_ids)
                if seq_id == 1
            ]

            if not context_token_indices:
                continue

            context_start = context_token_indices[0]
            context_end = context_token_indices[-1]

            # Check whether the COMPLETE answer is inside this window.
            #
            # If the answer starts before the first context token,
            # or ends after the last context token, this window
            # does not contain the complete answer.
            if offsets[context_start][0] > answer_start_char:
                continue

            if offsets[context_end][1] < answer_end_char:
                continue

            # Find the token containing the answer start.
            token_start = context_start

            while (
                token_start <= context_end
                and offsets[token_start][1] <= answer_start_char
            ):
                token_start += 1

            # Find the token containing the answer end.
            token_end = context_end

            while (
                token_end >= context_start
                and offsets[token_end][0] >= answer_end_char
            ):
                token_end -= 1

            # Safety check.
            if token_start > context_end or token_end < context_start:
                continue

            # Keep this window.
            keep_indices.append(feature_idx)
            start_positions.append(token_start)
            end_positions.append(token_end)

        # Keep only windows containing the complete answer.
        tokenized = {
            key: [
                values[i]
                for i in keep_indices
            ]
            for key, values in tokenized.items()
        }

        tokenized['start_positions'] = start_positions
        tokenized['end_positions'] = end_positions

        # Save bookkeeping data for string-level SQuAD evaluation
        if save_bookkeeping:
            # Extend (do NOT overwrite): .map() calls this function once per
            # batch, and the mapped dataset concatenates each batch's features
            # in call order, so appending keeps bookkeeping aligned with the
            # final dataset rows.
            _squad_eval_bookkeeping['feature_example_ids'].extend(
                examples['id'][sample_mapping[i]] for i in keep_indices
            )
            _squad_eval_bookkeeping['feature_offset_mappings'].extend(
                offset_mapping[i] for i in keep_indices
            )
            _squad_eval_bookkeeping['feature_contexts'].extend(
                examples['context'][sample_mapping[i]] for i in keep_indices
            )

            # Build references (one per unique example), deduped across batches
            seen = {
                ref['id']
                for ref in _squad_eval_bookkeeping['references']
            }
            for i in keep_indices:
                ex_id = examples['id'][sample_mapping[i]]
                if ex_id not in seen:
                    seen.add(ex_id)
                    _squad_eval_bookkeeping['references'].append({
                        'id': ex_id,
                        'answers': examples['answers'][sample_mapping[i]],
                    })

        return tokenized


    train_dataset = train_dataset.map(
        preprocess_squad,
        batched=True,
        remove_columns=train_dataset.column_names,
    )

    val_dataset = val_dataset.map(
        lambda examples: preprocess_squad(examples, save_bookkeeping=True),
        batched=True,
        remove_columns=val_dataset.column_names,
        # The bookkeeping is a SIDE EFFECT of this map call: never serve it
        # from the datasets cache, otherwise the function is skipped and the
        # bookkeeping stays empty/stale.
        load_from_cache_file=False,
    )

else:

    def tokenize_text(examples, indices):
        rng = random.Random(SEED)

        all_input_ids = []
        all_attention_masks = []
        all_labels = []

        for text, article_idx in zip(examples['text'], indices):

            # Tokenize the entire article without truncation.
            # Special tokens are added after chunking.
            article_tokens = tokenizer(
                text,
                truncation=False,
                add_special_tokens=False,
            )['input_ids']

            # Create non-overlapping 512-token chunks.
            chunks = [
                article_tokens[i:i + WIKI_MAX_LENGTH]
                for i in range(
                    0,
                    len(article_tokens),
                    WIKI_MAX_LENGTH,
                )
            ]

            # Keep only complete 512-token chunks.
            chunks = [
                chunk
                for chunk in chunks
                if len(chunk) == WIKI_MAX_LENGTH
            ]

            # Select up to N chunks randomly.
            num_chunks = min(
                NUM_CHUNKS_PER_WIKI_ARTICLE,
                len(chunks),
            )

            if num_chunks == 0:
                continue

            # Use a deterministic per-article RNG.
            article_rng = random.Random(
                SEED + article_idx
            )

            selected_chunks = article_rng.sample(
                chunks,
                num_chunks,
            )

            for chunk in selected_chunks:

                # Add special tokens in the same way
                # the XLM-R tokenizer normally does.
                input_ids = (
                    [tokenizer.bos_token_id]
                    + chunk
                    + [tokenizer.eos_token_id]
                )
                attention_mask = [1] * len(input_ids)

                all_input_ids.append(input_ids)
                all_attention_masks.append(attention_mask)
                all_labels.append(input_ids.copy())

        return {
            'input_ids': all_input_ids,
            'attention_mask': all_attention_masks,
            'labels': all_labels,
        }

    train_dataset = train_dataset.map(
        tokenize_text,
        batched=True,
        with_indices=True,
        remove_columns=train_dataset.column_names,
    )

    val_dataset = val_dataset.map(
        tokenize_text,
        batched=True,
        with_indices=True,
        remove_columns=val_dataset.column_names,
    )

In [ ]:
print("Total preprocessed training samples:", len(train_dataset))
print("Total preprocessed validation samples:", len(val_dataset))

# Training

In [ ]:
logger = logging.getLogger(__name__)

class LegamexTrainer(Trainer):
    def _save(self, output_dir=None, state_dict=None):
        output_dir = output_dir if output_dir is not None else self.args.output_dir
        if output_dir is None:
            return
        os.makedirs(output_dir, exist_ok=True)

        # Use our own lightweight saver instead of Trainer's default full state_dict dump
        self.model.save_pretrained(output_dir)

        # Still save processing_class/tokenizer + training args like Trainer normally would
        if self.processing_class is not None:
            self.processing_class.save_pretrained(output_dir)
        torch.save(self.args, os.path.join(output_dir, 'training_args.bin'))
        
    def _load_from_checkpoint(self, resume_from_checkpoint, model=None):
        # HF Trainer only calls this when resuming, but guard anyway so the
        # override honours the same contract as the base implementation
        # (resume_from_checkpoint=None means fresh training).
        if resume_from_checkpoint is None:
            return
        config_file = os.path.join(resume_from_checkpoint, 'legamex_config.json')
        if not os.path.isfile(config_file):
            # Fall back to original behaviour for regular checkpoints
            super()._load_from_checkpoint(resume_from_checkpoint, model=model)
            return

        # Load the checkpoint weights IN-PLACE into the model instance the
        # optimizer was built from. Do NOT reconstruct a replacement
        # LegamexLoraModel here: the optimizer passed via
        # optimizers=(optimizer, None) holds references to self.model's
        # parameter tensors, and Trainer.create_optimizer() is a no-op when
        # self.optimizer is already set (transformers 4.x/5.x). The previous
        # behaviour (self.model = LegamexLoraModel.from_pretrained(...))
        # rebound self.model to a brand-new instance, so backward populated
        # gradients on the new instance's tensors while optimizer.step()
        # updated the original instance's tensors (grad=None -> skipped).
        # Net effect: NOTHING trained after resume - val loss/F1/EM and every
        # gate statistic were frozen, and the learned gate offset never moved.
        target = model if model is not None else self.model
        if not isinstance(target, LegamexLoraModel):
            raise TypeError(
                f"LegameX checkpoint resume requires a LegamexLoraModel, got {type(target).__name__}."
            )
        target.load_legamex_state_dict(resume_from_checkpoint)

        # NOTE: trainer_state.json (global_step, best metric, ...) is restored
        # by Trainer._init_training_state() AFTER this hook runs, so self.state
        # is still the fresh state here. The gate warmup/offset schedules are
        # synced in _load_optimizer_and_scheduler() below, which runs after the
        # state restore and therefore sees the correct global_step.

        logger.info(f"Loaded LegameX weights from {resume_from_checkpoint} into the existing model instance")

    @staticmethod
    def _optimizer_state_shape(entry):
        """Best-effort shape of the parameter a saved optimizer-state entry belongs to.

        Works for plain torch optimizers (exp_avg/exp_avg_sq), bitsandbytes 8-bit
        optimizers (state1/state2, possibly wrapped in __bnb_optimizer_quant_state__)
        and momentum-style optimizers (momentum1/momentum2).
        """
        if not isinstance(entry, dict):
            return None
        for key in ('exp_avg', 'exp_avg_sq', 'state1', 'state2', 'momentum1', 'momentum2'):
            value = entry.get(key)
            if torch.is_tensor(value):
                return tuple(value.shape)
        quant = entry.get('__bnb_optimizer_quant_state__')
        if isinstance(quant, dict):
            for key in ('state1', 'state2'):
                value = quant.get(key)
                if torch.is_tensor(value):
                    return tuple(value.shape)
        return None

    def _align_optimizer_state(self, state, checkpoint_dir):
        """Reorder a saved optimizer state to the CURRENT optimizer's parameter order.

        `Optimizer.load_state_dict()` maps state entries to parameters by POSITION
        within each param group, but that position order follows
        `named_parameters()`, which is NOT stable across library versions.
        transformers 4.53.3 -> 5.17.0 moved XLMRobertaForQuestionAnswering's
        qa_outputs from last to first, so resuming checkpoint-4200 unaligned
        handed qa_outputs a shape-mismatched 8-bit quant state (garbage
        dequantization) and shifted every tfr matrix's moments - the loss went
        1.03 -> 48622 -> NaN within 100 steps.

        Alignment strategies, in order of preference:
        1. `optimizer_param_names.json` (written by _save_optimizer_and_scheduler()
           since this fix): exact, name-based remap.
        2. Shape-based FIFO within each group: assumes same-shaped parameters keep
           their relative order - true for the 4.53 -> 5.17 qa_outputs move, and
           verified against the real checkpoint-4200 optimizer.pt.

        Returns the state with each group's 'params' id list reordered to match
        this optimizer; raises if no faithful alignment is possible.
        """
        from collections import defaultdict, deque

        opt_groups = self.optimizer.param_groups
        saved_groups = state['param_groups']
        saved_state = state.get('state', {})

        if len(opt_groups) != len(saved_groups):
            raise RuntimeError(
                f"Optimizer has {len(opt_groups)} parameter groups but the checkpoint "
                f"state has {len(saved_groups)} - not the same optimizer, refusing to resume."
            )

        # 1. Exact name-based remap, if this checkpoint recorded parameter names
        saved_names = None
        names_path = os.path.join(checkpoint_dir, 'optimizer_param_names.json')
        if os.path.isfile(names_path):
            with open(names_path, 'r') as f:
                saved_names = json.load(f)

        id_to_name = None
        n_remapped = 0
        for gi, (cur_group, saved_group) in enumerate(zip(opt_groups, saved_groups)):
            cur_params = cur_group['params']
            saved_ids = list(saved_group['params'])
            if len(cur_params) != len(saved_ids):
                raise RuntimeError(
                    f"Optimizer group {gi} has {len(cur_params)} parameters but the "
                    f"checkpoint state has {len(saved_ids)} - refusing to resume."
                )

            new_ids = None
            if (saved_names is not None and gi < len(saved_names)
                    and len(saved_names[gi]) == len(saved_ids)):
                if id_to_name is None:
                    id_to_name = {id(p): n for n, p in self.model.named_parameters()}
                name_to_saved_id = dict(zip(saved_names[gi], saved_ids))
                if all(id_to_name.get(id(p)) in name_to_saved_id for p in cur_params):
                    new_ids = [name_to_saved_id[id_to_name[id(p)]] for p in cur_params]

            if new_ids is None:
                # 2. Shape-based FIFO fallback
                cur_shapes = [tuple(p.shape) for p in cur_params]
                saved_shapes = [self._optimizer_state_shape(saved_state.get(pid, {}))
                                for pid in saved_ids]
                if cur_shapes == saved_shapes:
                    continue  # already aligned
                if any(s is None for s in saved_shapes):
                    raise RuntimeError(
                        f"Optimizer group {gi} is misaligned with the checkpoint and the "
                        "saved state shapes cannot be determined, so it cannot be "
                        "realigned. Resume with a fresh optimizer (delete optimizer.pt) "
                        "instead."
                    )
                buckets = defaultdict(deque)
                for pid, shape in zip(saved_ids, saved_shapes):
                    buckets[shape].append(pid)
                new_ids = []
                for shape in cur_shapes:
                    if not buckets[shape]:
                        raise RuntimeError(
                            f"Optimizer group {gi} cannot be realigned: no saved state "
                            f"entry with shape {shape}. Resume with a fresh optimizer "
                            "(delete optimizer.pt) instead."
                        )
                    new_ids.append(buckets[shape].popleft())

            if new_ids != saved_ids:
                n_remapped += sum(1 for a, b in zip(new_ids, saved_ids) if a != b)
                saved_group['params'] = new_ids

        if n_remapped:
            logger.warning(
                f"Optimizer state was saved under a different parameter order "
                f"(likely a library version change); realigned {n_remapped} state "
                "entries before loading."
            )
        return state

    def _load_optimizer_and_scheduler(self, checkpoint):
        """Restore optimizer/scheduler state for LegameX checkpoints.

        Trainer calls this after creating the optimizer and scheduler, so it
        is the correct place to load optimizer.pt / scheduler.pt.
        """
        resume_from_checkpoint = checkpoint
        # NOTE: _inner_training_loop() calls this hook unconditionally, even
        # when trainer.train(resume_from_checkpoint=None) starts fresh. The
        # default HF implementation returns early on None — mirror that here.
        if resume_from_checkpoint is None:
            return
        config_file = os.path.join(resume_from_checkpoint, 'legamex_config.json')
        if not os.path.isfile(config_file):
            # Regular checkpoint: let the default implementation handle it
            super()._load_optimizer_and_scheduler(resume_from_checkpoint)
            return

        device = next(self.model.parameters()).device
        optimizer_path = os.path.join(resume_from_checkpoint, 'optimizer.pt')
        if (
            self.optimizer is not None and 
            isinstance(self.optimizer, torch.optim.Optimizer) and
            os.path.isfile(optimizer_path)
        ):
            optimizer_state = torch.load(optimizer_path, map_location=device)
            # load_state_dict maps moments to parameters BY POSITION within each
            # group, and that order follows named_parameters(), which changed
            # between transformers 4.53.3 (original run) and 5.17.0 (resume).
            # Align first - an unaligned load silently corrupted the 8-bit
            # moments and NaN'd training within 100 steps of resuming.
            optimizer_state = self._align_optimizer_state(
                optimizer_state, resume_from_checkpoint,
            )
            self.optimizer.load_state_dict(optimizer_state)
        scheduler_path = os.path.join(resume_from_checkpoint, 'scheduler.pt')
        if (
            self.lr_scheduler is not None and 
            isinstance(self.lr_scheduler, torch.optim.lr_scheduler.LambdaLR) and
            os.path.isfile(scheduler_path)
        ):
            self.lr_scheduler.load_state_dict(
                torch.load(scheduler_path, map_location=device)
            )
        scaler_path = os.path.join(resume_from_checkpoint, 'scaler.pt')
        if (
            getattr(self, 'scaler', None) is not None and 
            os.path.isfile(scaler_path)
        ):
            self.scaler.load_state_dict( # type: ignore
                torch.load(scaler_path, map_location=device)
            )

        # Sync the LegameX gate schedules (warmup / scheduled offset) with the
        # resumed step. This hook runs AFTER Trainer._init_training_state()
        # restored trainer_state.json into self.state, so global_step is
        # correct here (it is still 0 when _load_from_checkpoint runs).
        if hasattr(self.model, 'set_training_step'):
            self.model.set_training_step(self.state.global_step)

        logger.info(f"Restored optimizer/scheduler state from {resume_from_checkpoint}")

    def _save_optimizer_and_scheduler(self, output_dir):
        super()._save_optimizer_and_scheduler(output_dir)
        # Record the parameter names behind the optimizer's positional state so
        # a future resume can realign exactly, even if a library upgrade reorders
        # named_parameters() (see _align_optimizer_state).
        if self.optimizer is None:
            return
        try:
            id_to_name = {id(p): n for n, p in self.model.named_parameters()}
            names = [
                [id_to_name.get(id(p), f'<unknown>') for p in group['params']]
                for group in self.optimizer.param_groups
            ]
            with open(os.path.join(output_dir, 'optimizer_param_names.json'), 'w') as f:
                json.dump(names, f)
        except Exception as e:
            logger.warning(f"Could not save optimizer_param_names.json: {e}")

    def _load_best_model(self):
        """Load the best checkpoint's LegameX weights when load_best_model_at_end=True.

        The base implementation only looks for model.safetensors / adapter
        files, but our custom _save() writes legamex.pt + legamex_config.json
        instead, so the default path silently loads nothing (just a warning in
        the logs). Load the best checkpoint in-place into self.model - the same
        instance the optimizer trained - so trainer.model holds the best-F1
        weights after training.
        """
        if self.state.best_model_checkpoint is None:
            return
        config_file = os.path.join(self.state.best_model_checkpoint, 'legamex_config.json')
        if os.path.isfile(config_file):
            self.model.load_legamex_state_dict(self.state.best_model_checkpoint)
            logger.info(
                f"Loaded best LegameX weights from {self.state.best_model_checkpoint} "
                f"(score: {self.state.best_metric})."
            )
            return
        super()._load_best_model()


class GateStepCallback(TrainerCallback):
    """Updates the LegameX training step at every optimisation step."""
    def on_train_begin(self, args, state, control, **kwargs):
        # Guard against a silent failure mode: the optimizer must be bound to
        # THIS model instance's parameter tensors (object identity, not just
        # equal values). If they diverge - e.g. a resume path replaced
        # self.model with a reconstructed instance - every optimizer.step()
        # updates tensors that never receive gradients and training silently
        # freezes with perfectly flat metrics.
        model = kwargs.get('model')
        optimizer = kwargs.get('optimizer')
        if model is not None and optimizer is not None:
            opt = getattr(optimizer, 'optimizer', optimizer)  # unwrap Accelerate's AcceleratedOptimizer
            model_param_ids = {id(p) for p in model.parameters()}
            opt_param_ids = {id(p) for group in opt.param_groups for p in group['params']}
            if opt_param_ids and not opt_param_ids.issubset(model_param_ids):
                raise RuntimeError(
                    f"Optimizer is not bound to the current model instance: "
                    f"{len(opt_param_ids - model_param_ids)} optimizer parameter(s) are not "
                    "parameters of the model being trained. Training would silently update "
                    "the wrong tensors (flat loss/metrics). Rebuild the optimizer from this "
                    "model's named_parameters() before trainer.train()."
                )

    def on_step_end(self, args, state, control, **kwargs):
        model = kwargs['model']
        if hasattr(model, 'set_training_step'):
            model.set_training_step(state.global_step)
        # return control
    
class WandbGateStatsCallback(TrainerCallback):
    """Logs gate weight statistics at every optimisation step.

    IMPORTANT: *_grad_norm metrics are captured in on_pre_optimizer_step, not on_log.
    transformers >= 5.x inner-loop order per update step:
        backward -> clip_grad_norm -> on_pre_optimizer_step -> optimizer.step()
        -> model.zero_grad() -> on_step_end -> _maybe_log_save_evaluate -> on_log
    `model.zero_grad()` defaults to set_to_none=True (PyTorch >= 2), so by the
    time on_log fires every param.grad is None - reading .grad there always hits
    the `is None` branch and logs 0.0, even though gradients flow to the gate
    weights fine. on_pre_optimizer_step is the last event where .grad still holds
    the full (accumulated + clipped) gradient the optimizer is about to consume.
    """

    def __init__(self):
        # (gate_A, gate_B, offset_learned, tfr_A, tfr_B, n_modules, global_step)
        # from the latest optimizer step
        self._grad_norms = None

    @staticmethod
    def _aggregate_grad_norms(model):
        """Sum per-layer L2 norms of grads across LegamexLora modules.

        Covers gate A/B weights, the learned gate offset (gate.offset), and the
        transfer (tfr) LoRA A/B weights.

        Returns (gate_A, gate_B, offset_learned, tfr_A, tfr_B, n_modules).
        Layers whose .grad is None contribute 0.
        """
        lora_modules = [m for m in model.legamex_modules.values()
                        if hasattr(m, 'gate')]
        gate_A_grad_norm = 0.0
        gate_B_grad_norm = 0.0
        gate_offset_grad_norm = 0.0
        tfr_A_grad_norm = 0.0
        tfr_B_grad_norm = 0.0
        for m in lora_modules:
            gate_A_grad = m.gate.A.weight.grad
            gate_B_grad = m.gate.B.weight.grad
            gate_A_grad_norm += gate_A_grad.norm().item() if gate_A_grad is not None else 0.0
            gate_B_grad_norm += gate_B_grad.norm().item() if gate_B_grad is not None else 0.0

            # Learned gate offset (gate.offset, a scalar nn.Parameter)
            offset_grad = m.gate.offset.grad # type: ignore
            gate_offset_grad_norm += offset_grad.norm().item() if offset_grad is not None else 0.0

            # Transfer (tfr) LoRA A/B weights
            tfr_A_grad = m.lora.tfr.A.weight.grad # type: ignore
            tfr_B_grad = m.lora.tfr.B.weight.grad # type: ignore
            tfr_A_grad_norm += tfr_A_grad.norm().item() if tfr_A_grad is not None else 0.0
            tfr_B_grad_norm += tfr_B_grad.norm().item() if tfr_B_grad is not None else 0.0
        return (gate_A_grad_norm, gate_B_grad_norm, gate_offset_grad_norm,
                tfr_A_grad_norm, tfr_B_grad_norm, len(lora_modules))

    def on_pre_optimizer_step(self, args, state, control, model=None, **kwargs):
        # Fired after grad clipping, before optimizer.step()/zero_grad():
        # .grad still holds the gradient the optimizer is about to consume.
        if model is None or not hasattr(model, 'legamex_modules'):
            return
        grads = self._aggregate_grad_norms(model)
        if grads[-1]:
            self._grad_norms = grads + (state.global_step,)

    def on_log(self, args, state, control, model=None, logs=None, **kwargs):
        # Gather only the LegamexLora modules
        lora_modules = [m for m in model.legamex_modules.values() # type: ignore
                        if hasattr(m, 'gate')]  # or isinstance(m, LegamexLora)
        if not lora_modules:
            return

        # Gate scheduler alpha (same for all layers)
        first_module = lora_modules[0]
        alpha = first_module.gate_scheduler()

        # Aggregate gate weight statistics
        gate_mean = 0.0
        gate_comp_mean = 0.0
        gate_std = 0.0
        gate_raw_norm = 0.0
        gate_raw_mean = 0.0
        gate_raw_std = 0.0
        offset_sched_sum = 0.0
        offset_sched_sq_sum = 0.0
        offset_learned_sum = 0.0
        offset_learned_sq_sum = 0.0
        offset_total_sum = 0.0
        offset_total_sq_sum = 0.0
        n = len(lora_modules)

        for m in lora_modules:
            gate_raw_W = m.gate.B.weight @ m.gate.A.weight

            # Decompose the total gate offset into its scheduled and learned terms:
            # gate_offset_total(t) = gate_offset_scheduled(t) + gate_offset_learned
            gate_offset_sched = float(m.gate_offset_scheduler())
            gate_offset_learned = m.gate.offset.item() # type: ignore
            gate_offset_total = m.current_gate_offset()
            gate_offset_total = gate_offset_total.item() if torch.is_tensor(gate_offset_total) else float(gate_offset_total)

            gate_W = torch.sigmoid(gate_raw_W + gate_offset_total)
            gate_comp_W = 1.0 - gate_W
            
            # Gate statistics per layer
            gate_mean += gate_W.mean().item()
            gate_comp_mean += gate_comp_W.mean().item()
            gate_std += gate_W.std().item()
            
            # Raw gate statistics per layer
            gate_raw_norm += gate_raw_W.norm(p=2).item()
            gate_raw_mean += gate_raw_W.mean().item()
            gate_raw_std += gate_raw_W.std().item()
            
            # Gate offset sums and squared sums per layer (scheduled | learned | total)
            offset_sched_sum += gate_offset_sched
            offset_sched_sq_sum += gate_offset_sched * gate_offset_sched
            offset_learned_sum += gate_offset_learned
            offset_learned_sq_sum += gate_offset_learned * gate_offset_learned
            offset_total_sum += gate_offset_total
            offset_total_sq_sum += gate_offset_total * gate_offset_total

        # Gate offset statistics across all layers (scheduled | learned | total).
        # NOTE: the scheduled term is a scalar shared by every layer, so its
        # across-layer std is always 0.0 and its mean equals the scalar value.
        def _offset_mean_std(sq_sum, s):
            mean = s / n
            std = math.sqrt(max(sq_sum / n - mean ** 2, 0.0))
            return mean, std

        offset_sched_mean, offset_sched_std = _offset_mean_std(offset_sched_sq_sum, offset_sched_sum)
        offset_learned_mean, offset_learned_std = _offset_mean_std(offset_learned_sq_sum, offset_learned_sum)
        offset_total_mean, offset_total_std = _offset_mean_std(offset_total_sq_sum, offset_total_sum)

        # Gate A/B gradient norms: read the snapshot captured in on_pre_optimizer_step.
        # (.grad is already None here - zero_grad(set_to_none=True) ran before on_log.)
        if self._grad_norms is not None:
            (gate_A_grad_norm, gate_B_grad_norm, gate_offset_grad_norm,
             tfr_A_grad_norm, tfr_B_grad_norm, n_grad, _) = self._grad_norms
        else:
            # No optimizer step has run yet (e.g. logging_first_step): fall back to live grads
            (gate_A_grad_norm, gate_B_grad_norm, gate_offset_grad_norm,
             tfr_A_grad_norm, tfr_B_grad_norm, n_grad) = self._aggregate_grad_norms(model)

        wandb.log({
            'step': state.global_step,
            
            # Gate log
            'gate/mean': gate_mean / n,
            'gate/comp_mean': gate_comp_mean / n,
            'gate/std': gate_std / n,
            'gate/scheduler_alpha': alpha,
            
            # Raw gate log
            'gate_raw/norm': gate_raw_norm / n,
            'gate_raw/mean': gate_raw_mean / n,
            'gate_raw/std': gate_raw_std / n,
            'gate_raw/A_grad_norm': gate_A_grad_norm / n_grad if n_grad else 0.0,
            'gate_raw/B_grad_norm': gate_B_grad_norm / n_grad if n_grad else 0.0,
            
            # Gate offset log
            'gate_offset/scheduled_mean': offset_sched_mean,
            'gate_offset/scheduled_std': offset_sched_std,
            'gate_offset/learned_mean': offset_learned_mean,
            'gate_offset/learned_std': offset_learned_std,
            'gate_offset/total_mean': offset_total_mean,
            'gate_offset/total_std': offset_total_std,
            'gate_offset/learned_grad_norm': gate_offset_grad_norm / n_grad if n_grad else 0.0,
            
            # Transfer (tfr) LoRA log
            'lora_tfr/A_grad_norm': tfr_A_grad_norm / n_grad if n_grad else 0.0,
            'lora_tfr/B_grad_norm': tfr_B_grad_norm / n_grad if n_grad else 0.0,
        })

In [ ]:
# Calculate steps per epoch and max steps
steps_per_epoch = math.ceil(len(train_dataset) / (MINI_BATCH_SIZE * GRAD_ACCUM_STEPS))
max_steps = steps_per_epoch * NUM_EPOCHS
print("Steps per epoch:", steps_per_epoch)
print("Max steps:", max_steps)

In [ ]:
GATE_LR_MULTIPLIER = 50
GATE_OFFSET_LR_MULTIPLIER = 50

# The learned offset is registered as `self.gate.offset` (a scalar nn.Parameter on
# the gate ModuleDict), so its param name ends with '.gate.offset', while the gate
# weights are '.gate.A.weight' / '.gate.B.weight'. Match the offset FIRST and
# exclude it from the gate-weights group so it gets its own LR multiplier.
gate_offset_params = [p for n, p in legamex_model.named_parameters()
                      if n.endswith('.gate.offset') and p.requires_grad]
gate_params = [p for n, p in legamex_model.named_parameters()
               if '.gate.' in n and not n.endswith('.gate.offset') and p.requires_grad]
non_gate_params = [p for n, p in legamex_model.named_parameters()
                   if '.gate.' not in n and p.requires_grad]
param_groups = [
    {'params': non_gate_params, 'lr': LR, 'weight_decay': 0.01},
    {'params': gate_params, 'lr': LR * GATE_LR_MULTIPLIER, 'weight_decay': 0.0},
    {'params': gate_offset_params, 'lr': LR * GATE_OFFSET_LR_MULTIPLIER, 'weight_decay': 0.0},
]

# Set up the optimizer
# from torch.optim import AdamW
# optimizer = torch.optim.AdamW(param_groups, lr=LR, fused=True)
from bitsandbytes.optim.adamw import AdamW8bit
optimizer = AdamW8bit(param_groups)

In [ ]:
# Sanity check
for i, pg in enumerate(optimizer.param_groups):
    print(i, len(pg['params']), pg['lr'])

In [ ]:
# Initialize wandb
wandb.init(
    project="legamex",
    name=run_name,
)

# Load the SQuAD metric (string-level Exact Match and F1)
squad_metric = evaluate.load('squad')

# Compute metrics for SQuAD (string-level Exact Match and F1 via HF evaluate)
def compute_metrics_squad(eval_pred):
    start_logits, end_logits = eval_pred.predictions

    # Defensive check: bookkeeping must be feature-for-feature aligned with
    # the eval predictions. A mismatch means the preprocessing cell that
    # builds _squad_eval_bookkeeping did not run (or ran on a different
    # val set) - re-run it before training.
    n_features = len(start_logits)
    n_bookkeeping = len(_squad_eval_bookkeeping['feature_offset_mappings'])
    if n_features != n_bookkeeping:
        raise RuntimeError(
            f'Eval produced {n_features} features but the SQuAD bookkeeping '
            f'only has {n_bookkeeping}. Re-run the dataset preprocessing '
            f'cell so the bookkeeping is rebuilt for the full validation set.'
        )

    # For each feature, find the best span and reconstruct predicted text
    feature_predictions = []
    for feat_idx in range(len(start_logits)):
        s_log = start_logits[feat_idx]
        e_log = end_logits[feat_idx]
        offsets = _squad_eval_bookkeeping['feature_offset_mappings'][feat_idx]
        context = _squad_eval_bookkeeping['feature_contexts'][feat_idx]
        ex_id = _squad_eval_bookkeeping['feature_example_ids'][feat_idx]

        # Identify context tokens (non-zero offset range)
        context_tokens = [
            tok for tok, (start, end) in enumerate(offsets)
            if start != 0 or end != 0
        ]

        if not context_tokens:
            feature_predictions.append({
                'id': ex_id,
                'prediction_text': '',
                'score': -float('inf'),
            })
            continue

        # Find best span using top-k candidates
        n_best = 20
        k = min(n_best, len(context_tokens))
        top_starts = sorted(context_tokens, key=lambda t: s_log[t], reverse=True)[:k]
        top_ends = sorted(context_tokens, key=lambda t: e_log[t], reverse=True)[:k]

        best_score = -float('inf')
        best_start = context_tokens[0]
        best_end = context_tokens[0]

        for start_tok in top_starts:
            for end_tok in top_ends:
                if end_tok < start_tok:
                    continue
                score = float(s_log[start_tok] + e_log[end_tok])
                if score > best_score:
                    best_score = score
                    best_start = start_tok
                    best_end = end_tok

        # Reconstruct text
        start_char = offsets[best_start][0]
        end_char = offsets[best_end][1]
        pred_text = context[start_char:end_char]

        feature_predictions.append({
            'id': ex_id,
            'prediction_text': pred_text,
            'score': best_score,
        })

    # Group by example_id, pick best prediction per example
    example_predictions = {}
    for pred in feature_predictions:
        ex_id = pred['id']
        if ex_id not in example_predictions or pred['score'] > example_predictions[ex_id]['score']:
            example_predictions[ex_id] = pred

    predictions = [
        {'id': ex_id, 'prediction_text': pred['prediction_text']}
        for ex_id, pred in example_predictions.items()
    ]

    references = _squad_eval_bookkeeping['references']

    if not predictions or not references:
        return {'f1': 0.0, 'exact_match': 0.0}

    metrics = squad_metric.compute(predictions=predictions, references=references) or {
        'f1': 0.0,
        'exact_match': 0.0,
    }

    return {
        'f1': float(metrics.get('f1', 0.0)),
        'exact_match': float(metrics.get('exact_match', 0.0)),
    }

# Set up the trainer
training_args = TrainingArguments(
    remove_unused_columns=False,
    
    # Training arguments
    seed=SEED,
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    per_device_train_batch_size=MINI_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    max_steps = max_steps,
    warmup_steps = WARMUP_STEPS,
    learning_rate=LR,
    lr_scheduler_type='cosine',
    # optim='adamw_8bit',
    # max_grad_norm=1.0,
    max_grad_norm=1e9, # Temporarily disable the gradient clipping
    weight_decay=0.01,
    
    # Validation arguments
    eval_strategy='steps',
    eval_steps=EVAL_STEPS,
    
    # Logging arguments
    logging_strategy='steps',
    logging_steps=LOGGING_STEPS,
    # logging_first_step=True,
    report_to=['tensorboard', 'wandb'],
    
    # Saving arguments
    save_strategy='steps',
    save_steps=SAVE_STEPS,
    # save_total_limit=5, # 1 best + 4 recent checkpoints. WARN: It doesn't work
    
    # With load_best_model_at_end=True, your save_strategy will be ignored and default to eval_strategy.
    # So you will find one checkpoint at the end of each epoch.
    # https://discuss.huggingface.co/t/trainer-not-saving-after-save-steps/5464
    load_best_model_at_end=True,
    metric_for_best_model='f1' if TASK == 'squad' else 'eval_loss',
    greater_is_better=True if TASK == 'squad' else False,

    # Restore callback states (e.g. EarlyStoppingCallback's
    # early_stopping_patience_counter) from the checkpoint on resume.
    # Without this, the counter silently resets to 0 on every resume
    # (transformers 4.53.3 default), making early stopping more lenient.
    restore_callback_states_from_checkpoint=True,

    # run_name=run_name,
    output_dir=run_name,
    hub_model_id=hub_model_id,
    push_to_hub=True,
    hub_strategy='all_checkpoints',
    hub_always_push=True,
    
    # TEMP: Do not save and upload checkpoints for testing
    # save_strategy='no', 
    # load_best_model_at_end=False,
)
trainer = LegamexTrainer(
    model=legamex_model,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    args=training_args,
    optimizers=(optimizer, None),
    compute_metrics=compute_metrics_squad if TASK == 'squad' else None,
    # label_names=['labels'],
    callbacks = [
        EarlyStoppingCallback(
            early_stopping_patience=EARLY_STOPPING_PATIENCE,
            # early_stopping_threshold = 0.001,
        ),
        GateStepCallback(),
        WandbGateStatsCallback(),
    ],
)
trainer.label_names = label_names

In [ ]:
# Start training
trainer_stats = trainer.train(resume_from_checkpoint=resume_from_checkpoint)

In [ ]:
# Sanity check
with torch.no_grad():
    mean_list = []
    std_list = []
    
    # Inspect trainer.model - the instance that was actually trained and
    # evaluated (both resume and load_best_model_at_end load weights into it).
    # Note: after a resumed run the notebook variable `legamex_model` may be a
    # different object than the one Trainer trains; reading trainer.model is
    # always correct.
    for module_name, module in trainer.model.legamex_modules.items():
        if not hasattr(module, 'gate'):
            continue
        gate = module.gate
        gate_W = gate.B.weight @ gate.A.weight
        print(module_name.replace("__DOT__", "."))
        gate_offset = module.current_gate_offset() # scheduled + learned
        gate_offset = gate_offset.item() if torch.is_tensor(gate_offset) else float(gate_offset)
        mean = torch.sigmoid(gate_W + gate_offset).mean().item()
        std = torch.sigmoid(gate_W + gate_offset).std().item()
        mean_list.append(mean)
        std_list.append(std)
        print("gate_offset:", gate_offset)
        print("mean:", mean)
        print("std:", std)
        print()
    
    mean_all = torch.tensor(mean_list).mean().item()
    std_all = torch.tensor(std_list).mean().item()
    print("mean_all:", mean_all)
    print("std_all:", std_all)
    print()